In [21]:
%matplotlib tk
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, RadioButtons, CheckButtons
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
import os
import re
from pathlib import Path
import json
import time
import hashlib

# Functions

### Lorentzian

In [22]:
def lorentzian(x, A, x0, gamma, C):
    """
    Standard Lorentzian lineshape.
    A: Amplitude (Height above background)
    x0: Center position
    gamma: Half-width at half-maximum (HWHM)
    C: Constant background offset
    """
    return A * (gamma**2 / ((x - x0)**2 + gamma**2)) + C

### Finding Data

In [23]:
CACHE_DIR = "drive_index_cache_files"
if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

def get_all_files(root_path, force_update=False):
    """
    Returns a list of all file paths for the given root_path.
    Creates a specific, hashed cache file for this path in the background.
    """
    # 1. Generate a unique cache filename for THIS path
    # We hash the path string to create a safe filename (e.g. "cache_5d41402abc.json")
    path_str = str(root_path)
    path_hash = hashlib.md5(path_str.encode('utf-8')).hexdigest()
    cache_file = os.path.join(CACHE_DIR, f"cache_{path_hash}.json")

    # 2. Check if cache exists and we are not forcing an update
    if os.path.exists(cache_file) and not force_update:
        print(f"Loading file list from local cache: {cache_file}...")
        t0 = time.time()
        try:
            with open(cache_file, 'r') as f:
                file_list = json.load(f)
            print(f"Loaded {len(file_list)} files in {time.time()-t0:.2f}s")
            return file_list
        except Exception as e:
            print(f"Cache corrupted ({e}), rescanning...")

    # 3. If no cache or force_update, scan the drive
    print(f"Scanning drive (this may take a while)... {root_path}")
    t0 = time.time()
    
    root = Path(root_path)
    if not root.exists():
        print(f"Warning: Path not found: {root_path}")
        return []

    # Recursively find all files
    files = [p.as_posix() for p in root.rglob('*') if p.is_file()]
    
    print(f"Scan complete. Found {len(files)} files in {time.time()-t0:.2f}s")

    # 4. Save to the specific cache file
    print(f"Saving cache to {cache_file}...")
    with open(cache_file, 'w') as f:
        json.dump(files, f)
        
    return files

def find_file_cached(root_path, substring, force_refresh=False):
    # Get the list (manages cache automatically based on root_path)
    all_files = get_all_files(root_path, force_update=force_refresh)
    
    # Perform the search in memory
    matches = [f for f in all_files if substring in f.split('/')[-1]]
    
    return matches

### Loading Data

In [24]:
def load_data(name, path, manual_indices=None, sort=True):
    name_extension = name+".hdf5"
    if not path.startswith("Z:/"):
        path = "Z:/POBox/Jonas Gerber/05 - Measurements (Data)"+path
    filepath = os.path.join(path, name_extension)

    data = h5py.File(filepath, 'r')
    # Extract raw data to inspect
    raw_names = data['Data']['Channel names'][:]
    
    # Clean the names: 
    # 1. Select the first element of the tuple (the name)
    # 2. Decode bytes to string
    # 3. Strip whitespace
    channel_names = [x[0].decode('utf-8').strip() for x in raw_names]
    
    print("Found channels:", channel_names)
    # Output: ['V_SD Left C18 C6', 'PG Virtual', 'V_MG C17', ...]
    
    
    # 2. Define Flexible Patterns (Regex)
    # -----------------------------------
    # These patterns define what you are looking for.
    # .  = any character
    # * = zero or more times
    # |  = OR logic
    # ?  = optional character (good for _ vs space)
    patterns = {
        'V_SD':  r'V.?SD',                      # Matches "V_SD", "V SD"
        'V_PG':  r'PG.?Virtual|V.?PG|V.?CG',    # Matches "PG Virtual" OR "V_PG" OR "V_CG"
        'I_ACx': r'I.?SD.*AC.*x',               # Matches "I", (space/_), "SD", then "AC", then "x"
        'I_SD':  r'I.?SD.*?(C18|C17)',          # Matches "I_SD C18", "I SD C18" [potentially also finds AC!]
        'B':     r'.*Magnet.*|B',                 # Matches anything with "Magnet" in it
    }
    
    # 3. Helper Function to Get Index
    # -------------------------------
    def get_index(pattern, name_list):
        for i, name in enumerate(name_list):
            if re.search(pattern, name, re.IGNORECASE):
                return i
        if pattern not in ["I.?SD.*AC.*x", '.*Magnet.*|B']:
            raise ValueError(f"Could not find channel matching pattern: {pattern}")
        else:
            return -1

    # 4. Map Patterns to Indices
    # --------------------------
    indices = {key: get_index(pat, channel_names) for key, pat in patterns.items()}

    print("Mapped Indices:", indices)
    if manual_indices is not None:
        if len(manual_indices) == len(patterns.keys()):
            indices = {key: idx for key, idx in zip(patterns.keys(), manual_indices)}
            print("Overwrote Indices:", indices)
        else:
            print(f"Could not overwrite: manual_indices has length {len(manual_indices)}, needs length {len(patterns.keys())}")
    # Output: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 5}
    
    
    # 5. Extract Data Using Dynamic Indices
    # -------------------------------------
    # Now use the dictionary indices instead of hardcoded numbers
    dataset = data['Data']['Data']
    
    V_SD  = dataset[:, indices['V_SD'], :] * 2
    V_PG  = dataset[:, indices['V_PG'], :]
    if indices['I_ACx'] == -1:
        I_SD = dataset[:, indices['I_SD'], :]
        dx = V_PG[1, 0]-V_PG[0, 0]
        if dx == 0:
            dx = V_PG[0, 1]-V_PG[0, 0]
        print("dx=", dx)
        I_ACx = np.gradient(I_SD, dx, axis=1) * 1e-8
        print("Manually computed I_AC with np.gradient(I_SD,...).")
    else:
        I_ACx = dataset[:, indices['I_ACx'], :] * 1e-8
    if indices['B'] != -1:
        B = dataset[:, indices['B'], :]
        n_B_values = len(np.unique(B))
        print(f"Detected {n_B_values} unique B-field values: {np.unique(B)}")
        # # We need to only take every n_B_values-th slice to get a single B-field
        # V_SD  = V_SD[:, ::n_B_values]
        # V_PG  = V_PG[:, ::n_B_values]
        # I_ACx = I_ACx[:, ::n_B_values]

    # Convert raw data to physical units
    y_all = V_PG[:, :] # 0:101]  # Plunger Gate (V)
    x_all = V_SD[:, :] # 0:101]  # Energy (meV)
    z_all = 1e12 * I_ACx[:, :] # 0:101]  # pA
    # z_all = np.fliplr(z_all)  # to test weak hole

    is_pg_horizontal = abs(V_PG[0, 1] - V_PG[0, 0]) > 1e-9

    if is_pg_horizontal:
        # CASE 1: Standard Orientation (PG is fast axis/columns)
        print("Detected orientation: PG varies along columns.")
        x_centers = V_PG[0, :]
        y_centers = V_SD[:, 0]
        z_final = z_all  # No transpose needed
        
    else:
        # CASE 2: Transposed Orientation (PG is slow axis/rows)
        print("Detected orientation: PG varies along rows. Transposing data...")
        x_centers = V_PG[:, 0]   # Take the column (vertical variation)
        y_centers = V_SD[0, :]   # Take the row (horizontal variation)
        
        # CRITICAL: Transpose Z so the plotting code (pcolormesh) 
        # still sees X as horizontal and Y as vertical.
        z_final = z_all.T

    # ---------------------------------------------------------
    # 7. SORTING (Fixes pcolormesh warning)
    # ---------------------------------------------------------
    if sort:
        # Ensure x_centers is strictly increasing
        if np.any(np.diff(x_centers) <= 0):
            sort_x = np.argsort(x_centers)
            x_centers = x_centers[sort_x]
            z_final = z_final[:, sort_x]  # Reorder columns of data

        # Ensure y_centers is strictly increasing
        if np.any(np.diff(y_centers) <= 0):
            sort_y = np.argsort(y_centers)
            y_centers = y_centers[sort_y]
            z_final = z_final[sort_y, :]  # Reorder rows of data

    return name, x_centers, y_centers, z_final

# Choosing data

In [13]:
search_dir = "Z:/POBox/Jonas Gerber/05 - Measurements (Data)/EE04/"
name_substring = "C0062"

# files = list(Path(search_dir).rglob(f"*{name_substring}*.hdf5"))
files = find_file_cached(search_dir, name_substring, force_refresh=False)

if files:
    for file in files:
        print(file)
else:
    print("No files found.")

Scanning drive (this may take a while)... Z:/POBox/Jonas Gerber/05 - Measurements (Data)/EE04/
Scan complete. Found 553 files in 15.08s
Saving cache to drive_index_cache_files\cache_13bd4ff15a240c96cf1cdcb409115a40.json...
No files found.


In [25]:
name, x_centers, y_centers, z_all = load_data(name='C0062 4VBGm 1e Diamond MG', \
                                              path='/EE01/Fritz2025 Cooldown C/2025/07/Data_0716/', sort=False)
# works well

Found channels: ['V_SD Left C18 C6', 'PG Virtual', 'V_MG C17', 'V_PG Left C24', 'I_SD Left C18', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 5, 'I_SD': 4, 'B': -1}
Detected orientation: PG varies along columns.


In [19]:
name, x_centers, y_centers, z_all = load_data('C0193_8VBG_1e diamond', \
                                              '/EE01/Fritz2025 Cooldown C/2025/07/Data_0725/')
z_all = z_all[::-1, :]  # flip vertically to match expected orientation
# works well

Found channels: ['V_SD Left C18 C6', 'PG virtual', 'V_PG Left C24', 'I SD AC C18 AC x', 'I_SD Left C18']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 3, 'I_SD': 3, 'B': -1}
Detected orientation: PG varies along columns.


In [6]:
name, x_centers, y_centers, z_all = load_data('C1108 1h high res diamond', \
                                              '/EE01/Fritz2025 Cooldown C/2025/12/Data_1210/')
# works okay

Found channels: ['PG virtual', 'V_SD Left C18 GND', 'V_PG Left C24', 'V_PG Right C13', 'I SD Right C16', 'I SD C18 GND']
Mapped Indices: {'V_SD': 1, 'V_PG': 0, 'I_ACx': -1, 'I_SD': 5, 'B': -1}
dx= -0.0020000000000000018
Manually computed I_AC with np.gradient(I_SD,...).
Detected orientation: PG varies along rows. Transposing data...


In [ ]:
name, x_centers, y_centers, z_all = load_data('B0373 6VBG 1h diamond', \
                                              '/EE01/Fritz2025/2025/04/Data_0407/')
# works meeh

Found channels: ['V_SD Left C18 C6', 'V_PG Left C24', 'I_SD Left C18', 'I SD Right C16', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 4, 'I_SD': 2, 'B': -1}
Detected orientation: PG varies along columns.


In [104]:
name, x_centers, y_centers, z_all = load_data('B0638 6VBG 1h diamond', \
                                              '/EE01/Fritz2025/2025/05/Data_0521/')
# works okay with 2.9, 3, 2.84

Found channels: ['V_PG Left C24', 'V_SD Left C18 C6', 'I_SD Left C18', 'Value I SD C18 ACx', 'Value I SD C18 ACx', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_SD': 1, 'V_PG': 0, 'I_ACx': 3, 'I_SD': 2, 'B': -1}
Detected orientation: PG varies along rows. Transposing data...


In [ ]:
name, x_centers, y_centers, z_all = load_data('B0591 6VBG 1ediamond HQ', \
                                              '/EE01/Fritz2025/2025/05/Data_0514/') #, manual_indices=[0,1,3,3])
# works well

Found channels: ['V_SD Left C18 C6', 'PG-virtual', 'V_PG Left C24', 'I_SD Left C18', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 4, 'I_SD': 3, 'B': -1}
Detected orientation: PG varies along columns.


In [ ]:
name, x_centers, y_centers, z_all = load_data('B0451  6VBG 1h diamond lever arm', \
                                              '/EE01/Fritz2025/2025/04/Data_0427/')
# works well

Found channels: ['V_SD Left C18 C6', 'PG-virtual', 'V_PG Left C24', 'I_SD Left C18', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 4, 'I_SD': 3, 'B': -1}
Detected orientation: PG varies along columns.


In [ ]:
name, x_centers, y_centers, z_all = load_data('B0818 m5VBG 1e diamond check', \
                                              '/EE01/Fritz2025/2025/06/Data_0605/')
# difficult

Found channels: ['V_SD Left C18 C6', 'PG-virtual', 'V_PG Left C24', 'I_SD Left C18', 'Value I SD C18 ACx', 'Value I SD C18 ACx', 'I SD C18 AC x']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 4, 'I_SD': 3, 'B': -1}
Detected orientation: PG varies along columns.


In [100]:
name, x_centers, y_centers, z_all = load_data('B0902 m5VBG 1h Diamonds BField', \
                                              '/EE01/Fritz2025/2025/06/Data_0615/')

Found channels: ['PG-virtual', 'V_SD Left C18 C6', 'Magnet - B', 'V_PG Left C24', 'I_SD Left C18', 'Value I SD C18 ACx', 'Value I SD C18 ACx', 'I SD C18 AC x', 'I SD Right C16']
Mapped Indices: {'V_SD': 1, 'V_PG': 0, 'I_ACx': 5, 'I_SD': 4, 'B': 2}
Detected 13 unique B-field values: [0.    0.025 0.05  0.075 0.1   0.125 0.15  0.175 0.2   0.225 0.25  0.275
 0.3  ]
Detected orientation: PG varies along rows. Transposing data...


In [99]:
name, x_centers, y_centers, z_all = load_data('B149_Efe4 Diamond 1e BField', \
                                              '/EE04/BigMom2024/2024/12/Data_1206/')


Found channels: ['V_SD C8C17', 'V_CG1', 'B', 'I_SD_C17', 'ISD C8', 'I_MG_test']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': -1, 'I_SD': 3, 'B': 2}
dx= 0.000500000000000167
Manually computed I_AC with np.gradient(I_SD,...).
Detected 13 unique B-field values: [0.   0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1  0.11 0.12]
Detected orientation: PG varies along columns.


# Interactive SOC

In [ ]:
# --- 1. CONFIGURATION & INITIALIZATION ---
# Calculate data bounds automatically
d_x_min, d_x_max = np.min(x_centers), np.max(x_centers)
d_y_min, d_y_max = np.min(y_centers), np.max(y_centers)
d_z_max = np.nanmax(z_all)
x_span = d_x_max - d_x_min

# Set initial parameter guesses
init_params = {
    'L_min': d_x_min + 0.05 * x_span,
    'L_max': d_x_min + 0.30 * x_span,
    'R_min': d_x_max - 0.30 * x_span,
    'R_max': d_x_max - 0.05 * x_span,
    'height': 0.10 * d_z_max,
    'dist': 5,
    'prom': 0.05 * d_z_max,
}

# --- 2. FIGURE & PLOT SETUP ---
fig = plt.figure(figsize=(11, 7))
plt.subplots_adjust(bottom=0.40, top=0.85)

# Main Plot Area [left, bottom, width, height]
ax = fig.add_axes([0.15, 0.35, 0.65, 0.55])

# Background Data
z_original = z_all.copy()
plot_z = z_original if z_original.shape[0] == len(y_centers) else z_original.T

pcm = ax.pcolormesh(x_centers, y_centers, plot_z, cmap='binary', shading='auto')

# Colorbar [left, bottom, width, height]
cbar_ax = fig.add_axes([0.87, 0.40, 0.02, 0.45])
cbar = fig.colorbar(pcm, cax=cbar_ax, label=r"$I_\text{AC,x}$ (pA)")
# cbar_ax.tick_params(labelsize=10)

# Interactive Plot Elements
lines = {
    'L_gs': ax.plot([], [], color='cyan', linestyle='--', alpha=0.6, label='Left GS')[0],
    'L_es': ax.plot([], [], color='orange', linestyle='--', alpha=0.6, label='Left ES')[0],
    'R_gs': ax.plot([], [], color='magenta', linestyle='--', alpha=0.6, label='Right GS')[0],
    'R_es': ax.plot([], [], color='red', linestyle='--', alpha=0.6, label='Right ES')[0]
}

point_artists = { 'L_gs': None, 'L_es': None, 'R_gs': None, 'R_es': None }

# points = {
#     'L_gs': ax.errorbar([], [], yerr=[], fmt='o', color='cyan', ms=3, alpha=0.5, capsize=2),
#     'L_es': ax.errorbar([], [], yerr=[], fmt='o', color='orange', ms=3, alpha=0.5, capsize=2),
#     'R_gs': ax.errorbar([], [], yerr=[], fmt='o', color='magenta', ms=3, alpha=0.5, capsize=2),
#     'R_es': ax.errorbar([], [], yerr=[], fmt='o', color='red', ms=3, alpha=0.5, capsize=2)
# }

# Store raw points and sigmas for MC access
current_points = {
    'Lgs': None, 'Les': None, 'Rgs': None, 'Res': None,
    'Lgs_sig': None, 'Les_sig': None, 'Rgs_sig': None, 'Res_sig': None
}

markers = {
    'ref': ax.plot([], [], 'rx', ms=12, mew=2, label='Reference (E=0)')[0],
    'soc': ax.plot([], [], 'gx', ms=12, mew=2, label='SOC Intersect')[0]
}

log_text = ax.text(0.02, 0.02, "Ready.", transform=ax.transAxes, 
                   fontsize=9, verticalalignment='bottom', 
                   bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.9))

arrow_ann = ax.annotate('', xy=(0,0), xytext=(0,0), arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
text_ann = ax.text(0, 0, '', color='red', fontsize=12, fontweight='bold')

spans = {
    'left': ax.axvspan(init_params['L_min'], init_params['L_max'], color='cyan', alpha=0.1),
    'right': ax.axvspan(init_params['R_min'], init_params['R_max'], color='magenta', alpha=0.1)
}

ax.set_xlabel("Plunger Gate Voltage (V)", fontsize=10, labelpad=10)
ax.set_ylabel("Energy (meV)", fontsize=10, labelpad=10)
ax.legend(loc='upper left', fontsize=8, framealpha=0.9)

# --- 3. WIDGET LAYOUT ---
widgets = {}

def add_slider(label, valmin, valmax, valinit, pos, valstep=None):
    ax_s = fig.add_axes(pos)
    return Slider(ax_s, label, valmin, valmax, valinit=valinit, valstep=valstep)

# A. ZOOM CONTROLS (Top)
widgets['x_min'] = add_slider('X Min', d_x_min, d_x_max, d_x_min, [0.10, 0.96, 0.30, 0.02])
widgets['x_max'] = add_slider('X Max', d_x_min, d_x_max, d_x_max, [0.60, 0.96, 0.30, 0.02])
widgets['y_min'] = add_slider('Y Min', d_y_min, d_y_max, d_y_min, [0.10, 0.93, 0.30, 0.02])
widgets['y_max'] = add_slider('Y Max', d_y_min, d_y_max, d_y_max, [0.60, 0.93, 0.30, 0.02])

# B. ANALYSIS CONTROLS (Bottom)
widgets['L_min'] = add_slider('L ROI Min', d_x_min, d_x_max, init_params['L_min'], [0.15, 0.20, 0.30, 0.02])
widgets['L_max'] = add_slider('L ROI Max', d_x_min, d_x_max, init_params['L_max'], [0.15, 0.15, 0.30, 0.02])
widgets['R_min'] = add_slider('R ROI Min', d_x_min, d_x_max, init_params['R_min'], [0.60, 0.20, 0.30, 0.02])
widgets['R_max'] = add_slider('R ROI Max', d_x_min, d_x_max, init_params['R_max'], [0.60, 0.15, 0.30, 0.02])
widgets['height'] = add_slider('Height', -d_z_max, d_z_max, init_params['height'], [0.15, 0.10, 0.20, 0.02])
widgets['dist']   = add_slider('Dist', 1, len(x_centers) // 2, init_params['dist'], [0.45, 0.10, 0.20, 0.02], valstep=1)
widgets['prom']   = add_slider('Prom', 0, d_z_max, init_params['prom'], [0.75, 0.10, 0.15, 0.02])

# Buttons & Radio
ax_btn = fig.add_axes([0.08, 0.02, 0.18, 0.07])
btn_mc = Button(ax_btn, 'Run MC Error\nSimulation', color='lightgray', hovercolor='lightblue')

ax_abs = fig.add_axes([0.30, 0.02, 0.18, 0.07])
btn_abs = Button(ax_abs, 'Toggle Abs(Z)', color='lightgray', hovercolor='lightblue')
use_abs = False  # default state

ax_center = fig.add_axes([0.52, 0.02, 0.18, 0.07])
btn_center = Button(ax_center, 'Toggle Center(Z)', color='lightgray', hovercolor='lightblue')
use_center = False # default State

ax_median = fig.add_axes([0.74, 0.02, 0.18, 0.07])
btn_median = Button(ax_median, 'Toggle Median(Z)', color='lightgray', hovercolor='lightblue')
use_median = False # default State

ax_check = fig.add_axes([0.86, 0.28, 0.12, 0.05], frame_on=False)
check = CheckButtons(ax_check, ['Fit Peaks'], [False])
use_peak_fitting = False

ax_radio = fig.add_axes([0.02, 0.25, 0.10, 0.10])
radio = RadioButtons(ax_radio, ('Weak e-', 'Weak h+', 'Strong e-', 'Strong h+'), active=0)

# Store fit results for MC
current_fits = {'L': None, 'R': None}

# --- 4. CORE FUNCTIONS ---

# --- HELPER: CONSTRAINED LEAST SQUARES ---
def constrained_fit(Lgs, Les, L_sigs=None, R_sigs=None):
    """
    Fits two lines y1 = m*x1 + c1 and y2 = m*x2 + c2
    sharing the SAME slope 'm'.

    Parameters
    ----------
    Lgs : np.ndarray
        (N,2) array of ground state points
    Les : np.ndarray
        (N,2) array of excited state points
    L_sigs : np.ndarray, optional
        (N,) array of uncertainties (sigmas) for Y (left points)
    R_sigs : np.ndarray, optional
        (N,) array of uncertainties (sigmas) for Y (right points)

    Returns
    -------
    tuple or None
        If successful: (params, cov) where params = [m, c1, c2] and cov is the covariance matrix
        If failed: None
    """
    if len(Lgs) < 2 or len(Les) < 2:
        return None

    # Form matrices for solving: A * p = Y
    # p = [m, c1, c2]
    # Equation 1: m*x_gs + c1 + 0*c2 = y_gs
    # Equation 2: m*x_es + 0*c1 + c2 = y_es
    
    x1, y1 = Lgs[:, 0], Lgs[:, 1]
    x2, y2 = Les[:, 0], Les[:, 1]
    
    # Construct Design Matrix A
    # Col 0: x values (shared slope)
    # Col 1: Indicator for GS (1 for GS points, 0 for ES points)
    # Col 2: Indicator for ES (0 for GS points, 1 for ES points)
    
    A1 = np.vstack([x1, np.ones_like(x1), np.zeros_like(x1)]).T
    A2 = np.vstack([x2, np.zeros_like(x2), np.ones_like(x2)]).T
    A = np.vstack([A1, A2])
    Y = np.concatenate([y1, y2])
    
    # Solve with (Weighted) Least Squares
    # params = [slope, intercept_GS, intercept_ES]
    if L_sigs is not None and R_sigs is not None:
        w1 = 1.0 / (L_sigs**2 + 1e-15) # Add epsilon to avoid div by zero
        w2 = 1.0 / (R_sigs**2 + 1e-15)
        W_diag = np.concatenate([w1, w2])
        W = np.diag(W_diag)
        
        # Weighted Least Squares: Beta = (A^T W A)^-1 A^T W Y
        # Covariance: (A^T W A)^-1
        try:
            ATA = A.T @ W @ A
            ATWY = A.T @ W @ Y
            cov = np.linalg.inv(ATA)
            params = cov @ ATWY

            residuals = Y - (A @ params)
            dof = len(Y) - len(params) # Degrees of freedom
            chisq_reduced = np.sum((residuals**2) * W_diag) / dof if dof > 0 else 1.0

            # Scale the covariance matrix
            cov = cov * chisq_reduced

            return params, cov
        except:
            return None
    else:
        # Standard OLS fallback
        try:
            params, residuals, rank, s = np.linalg.lstsq(A, Y, rcond=None)
            cov = np.eye(3) * 1e-6
            if len(Y) > 3 and len(residuals) > 0:
                mse = residuals[0] / (len(Y) - 3)
                cov = mse * np.linalg.inv(A.T @ A)
            return params, cov
        except:
            return None
        
def safe_polyfit(pts, sigmas=None):
    try:
        if len(pts) > 2 and np.std(pts[:,0]) > 1e-9:
            if sigmas is not None:
                # np.polyfit takes weights = 1/sigma (it squares them internally)
                w = 1.0 / (sigmas + 1e-15)
                return np.polyfit(pts[:,0], pts[:,1], 1, w=w, cov=True)
            else:
                return np.polyfit(pts[:,0], pts[:,1], 1, cov=True)
    except: pass
    return None
    
# --- Helper to update slider ranges dynamically ---
def adjust_slider_range(slider, new_max):
    """Updates the max range of a slider to new_max.
       Clamps current value if it exceeds new_max."""
    # Temporarily disable events to prevent recursion loops
    old_eventson = slider.eventson
    slider.eventson = False
    
    # Update range
    slider.valmax = new_max
    slider.ax.set_xlim(slider.valmin, new_max)
    
    # Clamp value
    if slider.val > new_max:
        slider.set_val(new_max)
    
    # Restore events
    slider.eventson = old_eventson

def update_all(val):
    # 1. Update View & Spans
    x_lims = (widgets['x_min'].val, widgets['x_max'].val)
    y_lims = (widgets['y_min'].val, widgets['y_max'].val)
    ax.set_xlim(*x_lims); ax.set_ylim(*y_lims)
    
    l_roi = (widgets['L_min'].val, widgets['L_max'].val)
    r_roi = (widgets['R_min'].val, widgets['R_max'].val)
    spans['left'].set_x(l_roi[0]); spans['left'].set_width(l_roi[1]-l_roi[0])
    spans['right'].set_x(r_roi[0]); spans['right'].set_width(r_roi[1]-r_roi[0])

    col_mask = (x_centers >= x_lims[0]) & (x_centers <= x_lims[1])
    row_mask = (y_centers >= y_lims[0]) & (y_centers <= y_lims[1])

    # Check if we actually have data in the view (avoid empty slice errors)
    if np.any(col_mask) and np.any(row_mask):
        # Extract the subset of data visible on screen
        # plot_z is (Rows/Y, Cols/X), so we slice accordingly
        view_data = plot_z[np.ix_(row_mask, col_mask)]
        
        if view_data.size > 0:
            local_min = np.nanmin(view_data)
            local_max = np.nanmax(view_data)
            
            # 1. Update Colorbar
            pcm.set_clim(local_min, local_max)
            
            # 2. Update Slider Ranges
            # Use the larger of the abs(min) or abs(max) to set the bounds
            abs_max = max(abs(local_min), abs(local_max))
            if abs_max < 1e-12: abs_max = 1.0
            
            # Height: Update Min AND Max to handle negative centering
            # We temporarily disable events to avoid loops
            widgets['height'].eventson = False
            widgets['height'].valmin = -abs_max
            widgets['height'].valmax = abs_max
            widgets['height'].ax.set_xlim(-abs_max, abs_max)
            widgets['height'].eventson = True
            
            # Prominence: Scale to the new signal magnitude
            adjust_slider_range(widgets['prom'], abs_max)
            
            # Distance: Scale to the number of visible pixels
            # We use the 'width' of the view (number of columns) as the reference
            n_pixels = view_data.shape[1] 
            new_dist_max = max(1, n_pixels // 2) # Allow at least 2 peaks in view
            adjust_slider_range(widgets['dist'], new_dist_max)
    
    h, d, p = widgets['height'].val, widgets['dist'].val, widgets['prom'].val
    regime = radio.value_selected 

    # 2. Universal Point Finder
    def extract_pts(roi):
        eff_min = max(roi[0], x_lims[0])
        eff_max = min(roi[1], x_lims[1])
        col_mask = (x_centers >= eff_min) & (x_centers <= eff_max)
        row_mask = (y_centers >= y_lims[0]) & (y_centers <= y_lims[1])
        
        if np.sum(row_mask) < 5: return np.empty((0,2)), np.empty((0,2))
        
        cols = np.asarray(col_mask).nonzero()[0]
        y_active = y_centers[row_mask]
        y_indices = np.where(row_mask)[0]
        y_buf = (y_lims[1] - y_lims[0]) * 0.015

        dy_step = abs(y_active[1] - y_active[0])
        
        gs, es = [], []
        gs_sigs, es_sigs = [], []

        for c in cols:
            trace = plot_z[row_mask, c]
            peaks, props = find_peaks(trace, height=h, distance=d, prominence=p)

            # --- HELPER: FIT PEAK ---
            def fit_single_peak(peak_idx_local):
                # If fitting disabled, return pixel center and default sigma
                if not use_peak_fitting:
                    return y_active[peak_idx_local], dy_step / 2 # Default small sigma
                
                # Prepare Data Window
                window = 5 # Pixels each side
                idx_global = y_indices[peak_idx_local]
                
                # Bounds check
                idx_start = max(0, idx_global - window)
                idx_end = min(len(y_centers), idx_global + window)
                
                y_window = y_centers[idx_start:idx_end]
                z_window = plot_z[idx_start:idx_end, c]
                
                # Guess Parameters [A, x0, gamma, C]
                p0 = [
                    plot_z[idx_global, c],      # Height
                    y_active[peak_idx_local],   # Center
                    (y_window[-1]-y_window[0])/10, # Gamma
                    np.min(z_window)            # Offset
                ]
                
                bound_x0 = 3 * dy_step
                
                lower = [-np.inf, p0[1] - bound_x0, 0, -np.inf]
                upper = [np.inf,  p0[1] + bound_x0, np.inf, np.inf]
                
                try:
                    popt, pcov = curve_fit(lorentzian, y_window, z_window, p0=p0, bounds=(lower, upper), maxfev=1000)
                    perr = np.sqrt(np.diag(pcov))
                    
                    fit_x0 = popt[1]
                    fit_err = perr[1]
                    fit_gamma = popt[2]
                    
                    # Sanity Checks: 
                    # 1. Center must be within window
                    if not (y_window[0] < fit_x0 < y_window[-1]): return None, None
                    # 2. Width must be reasonable (not a spike < 1 pixel, not huge > window)
                    px_width = fit_gamma / abs(y_centers[1]-y_centers[0])
                    if px_width < 0.3 or px_width > window: return None, None
                    
                    return fit_x0, fit_err
                except:
                    return None, None # Fit failed
            # ------------------------

            if len(peaks) > 0:
                # ---------------------------------------------------
                # 1. IDENTIFY GROUND STATE (GS)
                # Assumption: GS is the brightest (highest Z) peak
                # ---------------------------------------------------
                gs_local_idx = np.argmax(props['peak_heights'])
                gs_idx = peaks[gs_local_idx]
                
                # Fit the GS Peak
                val_y_gs, val_sig_gs = fit_single_peak(gs_idx)
                
                # Store GS result
                if val_y_gs is not None:
                    gs.append([x_centers[c], val_y_gs])
                    gs_sigs.append(val_sig_gs)
                    gs_y_ref = val_y_gs # Use fitted Y for separation check
                else:
                    gs_y_ref = y_active[gs_idx] # Fallback to pixel Y

                # ---------------------------------------------------
                # 2. IDENTIFY EXCITED STATE (ES)
                # Constraint: Must be lower in energy (Y) than GS
                # AND separated by a minimum gap to avoid "shoulder" fitting
                # ---------------------------------------------------
                min_pixel_gap = 4  # Minimum pixels between GS and ES
                min_y_gap = min_pixel_gap * dy_step
                
                # Filter peaks to find ES candidates
                es_candidates = []
                for k in peaks:
                    if k == gs_idx: continue # Skip the GS itself
                    
                    # Check if this peak is "below" the GS (lower Y value)
                    # and sufficiently far away
                    if y_active[k] < (gs_y_ref - min_y_gap):
                        es_candidates.append(k)
                
                if len(es_candidates) > 0:
                    # Look up the Z-values (heights) in 'trace' directly
                    es_candidates = np.array(es_candidates)
                    es_heights = trace[es_candidates]
                    
                    # Pick the brightest candidate
                    best_cand_idx = np.argmax(es_heights)
                    es_idx = es_candidates[best_cand_idx]
                    
                    # Fit the ES Peak
                    val_y_es, val_sig_es = fit_single_peak(es_idx)
                    
                    if val_y_es is not None:
                        es.append([x_centers[c], val_y_es])
                        es_sigs.append(val_sig_es)

        return np.array(gs), np.array(es), np.array(gs_sigs), np.array(es_sigs)

    Lgs, Les, Lgs_sig, Les_sig = extract_pts(l_roi)
    Rgs, Res, Rgs_sig, Res_sig = extract_pts(r_roi)

    # --- SAVE DATA FOR MC ---
    current_points['Lgs'] = Lgs; current_points['Les'] = Les
    current_points['Rgs'] = Rgs; current_points['Res'] = Res
    current_points['Lgs_sig'] = Lgs_sig; current_points['Les_sig'] = Les_sig
    current_points['Rgs_sig'] = Rgs_sig; current_points['Res_sig'] = Res_sig

    def replot_points(key, pts, sigs, color, label):
        # Remove old artist if it exists
        if point_artists[key] is not None:
            point_artists[key].remove()
            point_artists[key] = None
            
        # Plot new one
        if len(pts) > 0:
            # If fitting is ON, show Error Bars. If OFF, just show dots (sig=None)
            yerr = sigs if use_peak_fitting else None
            
            # errorbar returns a Container. We store it to remove() later.
            container = ax.errorbar(pts[:,0], pts[:,1], yerr=yerr, 
                                    fmt='o', color=color, ms=3, alpha=0.6, capsize=3, label=label)
            point_artists[key] = container

    replot_points('L_gs', Lgs, Lgs_sig, 'cyan', 'Left GS')
    replot_points('L_es', Les, Les_sig, 'orange', 'Left ES')
    replot_points('R_gs', Rgs, Rgs_sig, 'magenta', 'Right GS')
    replot_points('R_es', Res, Res_sig, 'red', 'Right ES')
    
    # 3. Mode Selection & Line Fitting
    x_plot = np.linspace(x_lims[0], x_lims[1], 100)
    fit_L, fit_R = None, None
    valid_res = False
    
    # Clear all lines first
    for k in lines: lines[k].set_data([], [])

    if regime == 'Weak e-':
        fit_L = constrained_fit(Lgs, Les, Lgs_sig, Les_sig)
        fit_R = safe_polyfit(Rgs, Rgs_sig)
        if fit_L and fit_R:
            (mL, cLgs, cLes), _ = fit_L
            (mR, cRgs), _       = fit_R
            lines['L_gs'].set_data(x_plot, mL*x_plot + cLgs)
            lines['L_es'].set_data(x_plot, mL*x_plot + cLes)
            lines['R_gs'].set_data(x_plot, mR*x_plot + cRgs)
            x_ref = (cRgs - cLgs) / (mL - mR); y_ref = mL * x_ref + cLgs
            x_soc = (cRgs - cLes) / (mL - mR); y_soc = mL * x_soc + cLes
            valid_res = True

    elif regime == 'Weak h+':
        fit_L = safe_polyfit(Lgs, Lgs_sig)
        fit_R = constrained_fit(Rgs, Res, Rgs_sig, Res_sig)
        if fit_L and fit_R:
            (mL, cLgs), _       = fit_L
            (mR, cRgs, cRes), _ = fit_R
            lines['L_gs'].set_data(x_plot, mL*x_plot + cLgs)
            lines['R_gs'].set_data(x_plot, mR*x_plot + cRgs)
            lines['R_es'].set_data(x_plot, mR*x_plot + cRes)
            x_ref = (cRgs - cLgs) / (mL - mR); y_ref = mL * x_ref + cLgs
            x_soc = (cRes - cLgs) / (mL - mR); y_soc = mR * x_soc + cRes
            valid_res = True

    elif regime == 'Strong e-':
        fit_L = constrained_fit(Lgs, Res, Lgs_sig, Res_sig) 
        fit_R = safe_polyfit(Rgs, Rgs_sig)
        if fit_L and fit_R:
            (mL, cLgs, cLes), _ = fit_L
            (mR, cRgs), _       = fit_R
            lines['L_gs'].set_data(x_plot, mL*x_plot + cLgs)
            lines['R_es'].set_data(x_plot, mL*x_plot + cLes) 
            lines['R_gs'].set_data(x_plot, mR*x_plot + cRgs)
            x_ref = (cRgs - cLgs) / (mL - mR); y_ref = mL * x_ref + cLgs
            x_soc = (cRgs - cLes) / (mL - mR); y_soc = mL * x_soc + cLes
            valid_res = True

    elif regime == 'Strong h+':
        fit_L = safe_polyfit(Lgs, Lgs_sig)
        fit_R = constrained_fit(Rgs, Les, Rgs_sig, Les_sig) 
        if fit_L and fit_R:
            (mL, cLgs), _       = fit_L
            (mR, cRgs, cRes), _ = fit_R
            lines['L_gs'].set_data(x_plot, mL*x_plot + cLgs)
            lines['L_es'].set_data(x_plot, mR*x_plot + cRes) 
            lines['R_gs'].set_data(x_plot, mR*x_plot + cRgs)
            x_ref = (cRgs - cLgs) / (mL - mR); y_ref = mL * x_ref + cLgs
            x_soc = (cRes - cLgs) / (mL - mR); y_soc = mR * x_soc + cRes
            valid_res = True

    # 4. Final Updates
    current_fits['L'] = fit_L; current_fits['R'] = fit_R
    
    # # Points
    # points['L_gs'].set_data(Lgs[:,0], Lgs[:,1]) if len(Lgs) else points['L_gs'].set_data([],[])
    # points['L_es'].set_data(Les[:,0], Les[:,1]) if len(Les) else points['L_es'].set_data([],[])
    # points['R_gs'].set_data(Rgs[:,0], Rgs[:,1]) if len(Rgs) else points['R_gs'].set_data([],[])
    # points['R_es'].set_data(Res[:,0], Res[:,1]) if len(Res) else points['R_es'].set_data([],[])
    
    if valid_res:
        soc_val = np.abs(y_ref - y_soc) * 1000
        markers['ref'].set_data([x_ref], [y_ref])
        markers['soc'].set_data([x_soc], [y_soc])
        arrow_ann.set_visible(True); arrow_ann.xy = (x_soc, y_ref); arrow_ann.set_position((x_soc, y_soc))
        text_ann.set_text(f"{soc_val:.1f} µeV")
        text_ann.set_position((x_soc + (x_lims[1]-x_lims[0])*0.02, (y_ref+y_soc)/2))
        ax.set_title(f"[{name} | {regime}] $\\Delta E_{{SOC}}$=: {soc_val:.2f} µeV", color='black')
    else:
        markers['ref'].set_data([],[]); markers['soc'].set_data([],[])
        arrow_ann.set_visible(False); text_ann.set_text(""); ax.set_title(f"[{name}] Insufficient Data")

    fig.canvas.draw_idle()

# --- 5. MONTE CARLO ---
def run_mc(event):
    # Check if fits exist
    if current_fits['L'] is None or current_fits['R'] is None: 
        print("No fits available to simulate.")
        return
    
    # 1. Update Title and Force Draw
    ax.set_title("Running MC Simulation (Covariance Method)...", fontsize=12, color='blue')
    fig.canvas.draw()
    fig.canvas.flush_events() 
    
    regime = radio.value_selected
    N = 5000
    
    # --- HELPER: Sample Parameters from Covariance Matrix ---
    def sample_params(fit_result):
        params, cov = fit_result
        # Generate N random variations of the parameters [m, c1, c2...]
        # multivariate_normal handles the correlations (off-diagonals) automatically
        return np.random.multivariate_normal(params, cov, N)

    # 2. Sample Parameters
    # This uses the cov matrix calculated during the constrained_fit
    sL = sample_params(current_fits['L'])
    sR = sample_params(current_fits['R'])

    # Unpack parameters based on array shape
    # Shape is (N, 2) for simple line [m, c]
    # Shape is (N, 3) for constrained [m, c1, c2]
    
    # Left
    if sL.shape[1] == 3: mL, cL1, cL2 = sL[:,0], sL[:,1], sL[:,2]
    else:                mL, cL1, cL2 = sL[:,0], sL[:,1], None
        
    # Right
    if sR.shape[1] == 3: mR, cR1, cR2 = sR[:,0], sR[:,1], sR[:,2]
    else:                mR, cR1, cR2 = sR[:,0], sR[:,1], None

    # 3. Calculate SOC Intersection
    # (Vectorized calculation - extremely fast)
    if regime == 'Weak e-':
        # L is constr (mL, cLgs, cLes), R is simple (mR, cRgs)
        # Ref: Lgs(cL1) - Rgs(cR1). SOC: Les(cL2) - Rgs(cR1)
        x_ref = (cR1 - cL1)/(mL - mR); y_ref = mL * x_ref + cL1
        x_soc = (cR1 - cL2)/(mL - mR); y_soc = mL * x_soc + cL2

    elif regime == 'Weak h+':
        # L is simple (mL, cLgs), R is constr (mR, cRgs, cRes)
        x_ref = (cR1 - cL1)/(mL - mR); y_ref = mL * x_ref + cL1
        x_soc = (cR2 - cL1)/(mL - mR); y_soc = mR * x_soc + cR2

    elif regime == 'Strong e-':
        # L is constr (mL, cLgs, cLes_via_Res), R is simple (mR, cRgs)
        # Note: In Strong e-, "Les" points were actually Res points fitted to L slope
        x_ref = (cR1 - cL1)/(mL - mR); y_ref = mL * x_ref + cL1
        x_soc = (cR1 - cL2)/(mL - mR); y_soc = mL * x_soc + cL2

    elif regime == 'Strong h+':
        # L is simple (mL, cLgs), R is constr (mR, cRgs, cRes_via_Les)
        x_ref = (cR1 - cL1)/(mL - mR); y_ref = mL * x_ref + cL1
        x_soc = (cR2 - cL1)/(mL - mR); y_soc = mR * x_soc + cR2

    diffs = np.abs(y_ref - y_soc) * 1000 # Convert to µeV

    # 4. Final Update
    soc_mean = np.mean(diffs)
    soc_std = np.std(diffs)
    
    title_str = f"[{name} | {regime}] MC: $\\Delta E_{{SOC}}$ = {soc_mean:.2f} ± {soc_std:.2f} µeV"
    ax.set_title(title_str, color='green', fontweight='bold')
    print(f"[{name} | {regime}] MC Result: {soc_mean:.4f} +/- {soc_std:.4f} µeV (N={N})")
    
    fig.canvas.draw_idle()

def recalc_data():
    """Applies filters to z_original, respecting the current zoom for centering."""
    global plot_z
    
    # 1. Start with raw data
    temp_z = z_original.copy()
    
    # 2. Apply Centering (Targeting the VISIBLE region)
    if use_center or use_median:
        x_min, x_max = widgets['x_min'].val, widgets['x_max'].val
        y_min, y_max = widgets['y_min'].val, widgets['y_max'].val
        
        c_mask = (x_centers >= x_min) & (x_centers <= x_max)
        r_mask = (y_centers >= y_min) & (y_centers <= y_max)
        
        # Determine offset based on visible data
        if np.any(c_mask) and np.any(r_mask):
            view_subset = temp_z[np.ix_(r_mask, c_mask)]
            
            if use_center:
                # Geometric Midpoint ((Min+Max)/2)
                offset = (np.min(view_subset) + np.max(view_subset)) / 2
            else: 
                # Median (Most common value)
                offset = np.median(view_subset)
        else:
            # Fallback for empty view
            offset = np.median(temp_z) if use_median else (np.min(temp_z)+np.max(temp_z))/2
            
        temp_z = temp_z - offset
        
    # 3. Apply Absolute Value
    if use_abs:
        temp_z = np.abs(temp_z)
        cbar.set_label(r"$\left| I_\text{AC,x} \right|$ (pA)")
    else:
        cbar.set_label(r"$I_\text{AC,x}$ (pA)")
        
    # 4. Update Global State
    plot_z = temp_z
    
    # 5. Refresh Plot
    pcm.set_array(plot_z.ravel())
    
    # Trigger the standard update (for line fits and colorbar)
    update_all(None)

def toggle_abs(event):
    global use_abs
    use_abs = not use_abs
    btn_abs.color = 'orange' if use_abs else 'lightgray'
    btn_abs.label.set_text('Abs(Z) ON' if use_abs else 'Toggle Abs(Z)')
    recalc_data()

def toggle_center(event):
    global use_center, use_median
    
    # Toggle self
    use_center = not use_center
    
    # If turning ON, force Median OFF
    if use_center:
        use_median = False
        btn_median.color = 'lightgray'
        btn_median.label.set_text('Toggle Median(Z)')
        
    btn_center.color = 'orange' if use_center else 'lightgray'
    btn_center.label.set_text('Center(Z) ON' if use_center else 'Toggle Center(Z)')
    recalc_data()

def toggle_median(event):
    global use_center, use_median
    
    # Toggle self
    use_median = not use_median
    
    # If turning ON, force Center OFF
    if use_median:
        use_center = False
        btn_center.color = 'lightgray'
        btn_center.label.set_text('Toggle Center(Z)')
        
    btn_median.color = 'orange' if use_median else 'lightgray'
    btn_median.label.set_text('Median ON' if use_median else 'Toggle Median(Z)')
    recalc_data()

def toggle_fit(label):
    global use_peak_fitting
    use_peak_fitting = not use_peak_fitting
    update_all(None)

# --- 6. PRINT CURRENT PARAMETERS ON CLOSE ---
def on_close(event):
    print("\n" + "="*40 + "\n      SAVED PARAMETERS\n" + "="*40)
    print(f"ax.set_xlim([{widgets['x_min'].val:.4f}, {widgets['x_max'].val:.4f}])")
    print(f"ax.set_ylim([{widgets['y_min'].val:.4f}, {widgets['y_max'].val:.4f}])")
    print(f"roi_L = ({widgets['L_min'].val:.4f}, {widgets['L_max'].val:.4f})")
    print(f"roi_R = ({widgets['R_min'].val:.4f}, {widgets['R_max'].val:.4f})")
    print(f"peaks = {{'h': {widgets['height'].val:.4f}, 'd': {int(widgets['dist'].val)}, 'p': {widgets['prom'].val:.4f}}}")
    print("="*40 + "\n")

# Connect Callbacks
for w in widgets.values(): w.on_changed(update_all)
btn_mc.on_clicked(run_mc)
btn_abs.on_clicked(toggle_abs)
btn_center.on_clicked(toggle_center)
btn_median.on_clicked(toggle_median)
check.on_clicked(toggle_fit)
radio.on_clicked(update_all)
fig.canvas.mpl_connect('close_event', on_close)

# Start
update_all(None)
plt.show(block=True)

C:\Users\jonat\AppData\Local\Temp\ipykernel_31968\2157036488.py:30: UserWarning: The input coordinates to pcolormesh are interpreted as cell centers, but are not monotonically increasing or decreasing. This may lead to incorrectly calculated cell edges, in which case, please supply explicit cell edges to pcolormesh.
  pcm = ax.pcolormesh(x_centers, y_centers, plot_z, cmap='binary', shading='auto')


[C0062 4VBGm 1e Diamond MG | Weak e-] MC Result: 144.9809 +/- 1890.3926 µeV (N=5000)

      SAVED PARAMETERS
ax.set_xlim([-0.5151, -0.4501])
ax.set_ylim([-0.5000, 0.5000])
roi_L = (-0.5119, -0.4956)
roi_R = (-0.4696, -0.4534)
peaks = {'h': 7.6905, 'd': 5, 'p': 3.8452}

